In [2]:
import tapas.datasets
import tapas.generators
import tapas.threat_models
import tapas.attacks
import tapas.report

# Some fancy displays when training/testing.
import tqdm

from sklearn.ensemble import RandomForestClassifier

print("Loading dataset...")
# We attack the 1% Census Microdata file, available at:
#  https://www.ons.gov.uk/census/2011census/2011censusdata/censusmicrodata/microdatateachingfile
# We have created a .json description file, so that tapas.Dataset.read can load both.
data = tapas.datasets.TabularDataset.read(
    "data/2011", label="Census"
)
# Select an arbitrary target record (that is unique), and remove it from the dataset.
target_record = data.get_records([1])
# This step is important to ensure that the target record is not (incorrectly) added
# to training datasets that are not supposed to contain it.
data.drop_records([1], in_place=True)

# We attack the (trivial) Raw generator, which outputs its training dataset.
generator = tapas.generators.Raw()

# We now define the threat model: what is assumed that the attacker knows.
# We first define what the attacker knows about the dataset. Here, we assume
# they have access to an auxiliary dataset from the same distribution.
data_knowledge = tapas.threat_models.AuxiliaryDataKnowledge(
    # The attacker has access to 50% of the data as auxiliary information.
    # This information will be used to generate training datasets.
    data,
    auxiliary_split=0.5,
    # The attacker knows that the real dataset contains 5000 samples. This thus
    # reflects the attacker's knowledge about the real data.
    num_training_records=5000,
)

# We then define what the attacker knows about the synthetic data generator.
# This would typically be black-box knowledge, where they are able to run the
# (exact) SDG model on any dataset that they choose, but can only observe
# (input, output) pairs and not internal parameters.
sdg_knowledge = tapas.threat_models.BlackBoxKnowledge(
    generator,
    # The attacker also specifies the size of the output dataset. In practice,
    # use the size of the published synthetic dataset.
    num_synthetic_records=5000,
)

# Now that we have defined the attacker's knowledge, we define their goal.
# We will here focus on a membership inference attack on a random record.
threat_model = tapas.threat_models.TargetedMIA(
    attacker_knowledge_data=data_knowledge,
    # We here select the first record, arbitrarily.
    target_record=target_record,
    attacker_knowledge_generator=sdg_knowledge,
    # These are mostly technical questions. They inform how the attacker will
    # be trained, but are not impactful changes of the threat model.
    #  - do we generate pairs (D, D U {target}) to train the attack?
    generate_pairs=True,
    #  - do we append the target to the dataset, or replace a record by it?
    replace_target=True,
    # (Optional) nice display for training and testing.
    iterator_tracker=tqdm.tqdm,
)

# Next step: initialise an attacker. Here, we just apply the GroundHog attack
# with standard parameters (from Stadler et al., 2022).
attacker = tapas.attacks.GroundhogAttack()

Loading dataset...


c:\Users\arthe\repo\Benchmarking-of-Tabular-Synthetic-Data-Generation\.conda\lib\site-packages\tapas\datasets\dataset.py:37: DtypeWarning: Columns (3,4,5,6,7,8,9,10,11,12,13,14,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(fp, header=validate_header(fp, cnames), dtype=dtypes, index_col=None, names=cnames)


In [3]:
print("Training the attack...")
# Having defined all the objects that we need, we can train the attack.
attacker.train(
    # The TargetedMIA threat model is a TrainableThreatModel: it defines a method
    #  to generate training samples (synthetic_dataset, target_in_real_dataset).
    # This is why the threat model is passed to train the attacker.
    threat_model,
    # This is the number of training pairs generated by the threat model to
    # train the attacker.
    num_samples=1000,
)

print("Testing the attack...")
# The attack is trained! Evaluate it within the test model.
# [explain why we split this way.]
attack_summary = threat_model.test(attacker, num_samples=100)

# Output nice, printable metrics that evaluate the attack.
metrics = attack_summary.get_metrics()
print("Results:\n", metrics.head())

# Publish human-readable graphs for this attack.
report = tapas.report.BinaryAIAttackReport(
    [attack_summary], metrics=["accuracy", "privacy_gain", "auc"]
)
report.publish("groundhog-census")

Training the attack...


100%|██████████| 1000/1000 [00:00<00:00, 1576.02it/s]


ValueError: 1 is not in list